# 02 — Downloading the hearing transcripts and their MODS metadata

**Purpose.** This notebook retrieves the two files each hearing package needs —
the MODS metadata record and the HTML transcript — for every package
`01_enumerate_hearings.ipynb` listed. It records an outcome for every single
package, so at the end "how many are missing" is a subtraction against the
manifest rather than an inference from whatever happens to be on disk.

That property is the reason this notebook exists as a separate step. The
collection this project replaces discovered and downloaded in one pass, so it
had nothing to check itself against: when a request failed it logged a line and
moved on, and the absence became invisible the moment the next step globbed the
filesystem for whatever files were there. Roughly 12 percent of the corpus went
missing that way and was not noticed for eight months.

**Inputs.**

| Source | Path | Contents |
|---|---|---|
| Package manifest | `GOVINFO_MANIFEST / packages_all.parquet` | One row per package, from `01_enumerate_hearings.ipynb`. |
| GovInfo API | `GET /packages/{packageId}/mods` | MODS XML: committee, members who spoke, witnesses, dates. |
| GovInfo API | `GET /packages/{packageId}/htm` | The hearing transcript as HTML. |

**Method.**

1. Read the manifest and treat it as the work list. Every row is attempted;
   nothing is skipped on the basis of what is already on disk without that skip
   being recorded as such.
2. Write to `GOVINFO_RAW / {year} / {mods,html} /`, partitioned by the same
   `dateIssued` year the manifest uses, so a single year can be re-collected
   without touching the rest.
3. Skip a file that already exists and is non-empty, and record the skip. Reruns
   are cheap and the notebook is safe to interrupt — which matters for a job
   that issues tens of thousands of requests over a couple of hours.
4. Record one JSON line per package per file kind in
   `GOVINFO_STATUS / fetch_{year}.jsonl`: `ok`, `skipped`, `missing` (the API
   returned 404 — some packages genuinely have no HTML rendition), or `error`
   with the message. The status log is the audit trail; the filesystem is not.
5. Let a transient failure retry inside `lib/govinfo.py` and, if the budget is
   spent, record it as `error` and continue to the next package. This is the one
   place the collector deliberately continues past a failure, and it is safe
   only because the failure lands in the status log and the verification cell
   refuses to pass while any `error` rows remain.

**Outputs.**

| File | Path | Contents |
|---|---|---|
| MODS records | `GOVINFO_RAW / {year} / mods / {packageId}.xml` | One per package. |
| Transcripts | `GOVINFO_RAW / {year} / html / {packageId}.htm` | One per package that has an HTML rendition. |
| Fetch status | `GOVINFO_STATUS / fetch_{year}.jsonl` | One line per package-file attempt. |
| Status summary | `GOVINFO_STATUS / fetch_summary.parquet` | Every status line, concatenated. |

**Notes.**

The corpus is roughly 34,000 packages, so about 68,000 requests. At the client's
default 10 requests per second that is close to two hours. GovInfo allows 40 per
second for a registered key; the collector stays well under its own allowance
because there is no deadline here and the API is a shared public service.

A package with no HTML rendition is a real feature of the collection, not a
failure — scanned older hearings sometimes have only PDF. Those are recorded as
`missing` and counted in the verification cell, which is what separates "GovInfo
does not have it" from "we failed to get it". The previous collector could not
make that distinction because it recorded neither.

## Setup

Same two modules as `01`. `KINDS` maps the API's file-kind token to the
extension written on disk: GovInfo serves the transcript at `/htm` but the file
is HTML, and the MODS record at `/mods` but the file is XML, so the on-disk
names say what the files are rather than which endpoint produced them.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, "../lib")

import pandas as pd

from govinfo import GovInfoClient, GovInfoError, MissingFile, load_api_key
from paths import GOVINFO_MANIFEST, GOVINFO_RAW, GOVINFO_STATUS, govinfo_year_dir

# API file-kind token -> (subdirectory, extension on disk)
KINDS = {"mods": ("mods", ".xml"), "htm": ("html", ".htm")}

GOVINFO_STATUS.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(GOVINFO_MANIFEST / "packages_all.parquet")
client = GovInfoClient(api_key=load_api_key(env_file=".env"))

print(f"{len(manifest):,} packages to fetch, {len(KINDS)} files each "
      f"= {len(manifest) * len(KINDS):,} requests at most")
print(f"years {manifest['query_year'].min()}-{manifest['query_year'].max()}")

## Step 1 — One package, one recorded outcome

`fetch_one` is where the contract lives. Every path through it returns a status
dictionary; there is no path that returns nothing, and no exception escapes it
except a programming error. The four statuses are the four things that can
actually happen, and keeping them distinct is what lets the verification cell
tell a genuine gap in GovInfo's holdings from a failure of this notebook.

In [ ]:
def fetch_one(package_id: str, year: int, kind: str) -> dict:
    """Fetch one rendition of one package, returning what happened.

    Returns a record with status in {ok, skipped, missing, error}:
      ok      wrote the file
      skipped the file was already on disk and non-empty
      missing GovInfo returned 404 — this package has no rendition of this kind
      error   transient failure that outlived the client's retry budget
    """
    subdir, extension = KINDS[kind]
    target = govinfo_year_dir(year, subdir) / f"{package_id}{extension}"
    record = {
        "packageId": package_id,
        "year": year,
        "kind": kind,
        "path": str(target),
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

    if target.exists() and target.stat().st_size > 0:
        return {**record, "status": "skipped", "bytes": target.stat().st_size}

    try:
        payload = client.fetch_file(package_id, kind)
    except MissingFile:
        return {**record, "status": "missing", "bytes": 0}
    except GovInfoError as exc:
        return {**record, "status": "error", "bytes": 0, "message": str(exc)[:300]}

    # Write to a temporary name and rename, so an interrupted write cannot leave
    # a truncated file that the skip check above would later accept as complete.
    tmp = target.with_suffix(target.suffix + ".partial")
    tmp.write_bytes(payload)
    tmp.replace(target)
    return {**record, "status": "ok", "bytes": len(payload)}

## Step 2 — Walk the manifest, year by year

Status lines are appended to the year's JSONL as they happen rather than
accumulated in memory and written at the end. If this notebook is interrupted —
a wall-clock limit, a dropped connection, a cancelled session — the record of
what was already done survives, and re-running resumes instead of restarting.
The `done` set that opens each year is what makes the rerun cheap: it is read
from the status log, not from the filesystem, so a file that was written but
never recorded is fetched again rather than assumed good.

One line of output per year, not a progress bar. Notebooks here are committed
with their outputs, and a bar that redraws tens of thousands of times leaves
either a wall of carriage returns or an ipywidgets blob in the committed file.
The per-year line carries the same information a reader needs afterwards.

In [ ]:
counts = {"ok": 0, "skipped": 0, "missing": 0, "error": 0}
started = datetime.now(timezone.utc)

for year, group in manifest.groupby("query_year", sort=True):
    status_path = GOVINFO_STATUS / f"fetch_{year}.jsonl"
    done = set()
    if status_path.exists():
        for line in status_path.read_text().splitlines():
            row = json.loads(line)
            if row["status"] in {"ok", "skipped", "missing"}:
                done.add((row["packageId"], row["kind"]))

    todo = [(pid, kind) for pid in group["packageId"] for kind in KINDS
            if (pid, kind) not in done]
    if not todo:
        print(f"{year}  {len(group):>6,} packages | already complete")
        continue

    year_counts = {"ok": 0, "skipped": 0, "missing": 0, "error": 0}
    with status_path.open("a") as fh:
        for package_id, kind in todo:
            record = fetch_one(package_id, int(year), kind)
            fh.write(json.dumps(record) + "\n")
            fh.flush()
            year_counts[record["status"]] += 1
            counts[record["status"]] += 1

    elapsed = (datetime.now(timezone.utc) - started).total_seconds()
    print(f"{year}  {len(group):>6,} packages | " +
          " ".join(f"{k}={v:,}" for k, v in year_counts.items() if v) +
          f" | {elapsed / 60:.0f} min elapsed")

print()
print("totals:", " ".join(f"{k}={v:,}" for k, v in counts.items()))
print(f"{client.n_requests:,} API requests, {client.n_retries:,} retries")

## Step 3 — Consolidate the status log

Every JSONL line from every year, in one parquet. This is the table the
verification cell reads and the one a later stage should consult before assuming
a transcript is on disk.

In [ ]:
status_frames = [
    pd.read_json(path, lines=True)
    for path in sorted(GOVINFO_STATUS.glob("fetch_*.jsonl"))
]
status = pd.concat(status_frames, ignore_index=True)
status.to_parquet(GOVINFO_STATUS / "fetch_summary.parquet", index=False)

pivot = status.pivot_table(
    index="year", columns="status", values="packageId", aggfunc="count", fill_value=0
)
print(pivot.to_string())
print()
print(f"total bytes written: {status.loc[status['status'] == 'ok', 'bytes'].sum() / 1e9:.1f} GB")

## Verification

Five checks. The manifest from `01` is the denominator for all of them, which is
the whole reason the two notebooks are separate.

1. **No `error` rows.** An error means a transient failure outlived the retry
   budget. Those are recoverable by re-running this notebook, so a non-zero
   count is a stop condition, not a finding — the corpus is not complete yet.
2. **Every manifest package has a MODS record.** MODS carries the committee and
   the list of members who spoke, and the topic-extraction stage cannot operate
   on a hearing without it. A gap here silently shrinks that stage's input.
3. **Every package has an outcome for every file kind.** Attempted and recorded
   are not the same thing; this catches a package the loop never reached.
4. **Files on disk match the `ok` and `skipped` rows.** Guards against the
   status log and the filesystem drifting apart — a file deleted by hand, or a
   write that reported success without landing.
5. **The `missing` rate is reported, not asserted.** Some older hearings have no
   HTML rendition at all. That is a fact about GovInfo, so the check prints the
   rate and the affected years rather than failing; what would be a defect is
   not knowing the number.

In [ ]:
errors = status.loc[status["status"] == "error"]
assert len(errors) == 0, (
    f"{len(errors)} packages failed after retries — re-run this notebook to resume. "
    f"first: {errors.iloc[0]['packageId'] if len(errors) else ''}"
)
print("1) no unresolved fetch errors")

have_mods = set(status.loc[(status["kind"] == "mods")
                           & status["status"].isin(["ok", "skipped"]), "packageId"])
missing_mods = set(manifest["packageId"]) - have_mods
assert not missing_mods, (
    f"{len(missing_mods)} packages have no MODS record; topic extraction reads "
    f"committee and speaker lists from it. e.g. {sorted(missing_mods)[:5]}"
)
print(f"2) all {len(have_mods):,} packages have a MODS record")

expected = len(manifest) * len(KINDS)
recorded = len(status)
assert recorded == expected, f"expected {expected:,} outcomes, log holds {recorded:,}"
print(f"3) {recorded:,} outcomes recorded, one per package per file kind")

# One directory listing per folder rather than one stat() per row: 70k stat
# calls against a shared parallel filesystem is minutes, a few listdirs is not.
present = set()
for subdir, _ in KINDS.values():
    for year_dir in sorted(GOVINFO_RAW.glob(f"*/{subdir}")):
        present.update(str(p) for p in year_dir.iterdir())

claimed_paths = set(status.loc[status["status"].isin(["ok", "skipped"]), "path"])
absent = claimed_paths - present
orphan = present - claimed_paths
assert not absent, f"log claims {len(absent):,} files that are not on disk, e.g. {sorted(absent)[:3]}"
assert not orphan, f"{len(orphan):,} files on disk have no status row, e.g. {sorted(orphan)[:3]}"
print(f"4) {len(claimed_paths):,} files on disk match the status log exactly")

missing = status.loc[status["status"] == "missing"]
print(f"5) {len(missing):,} renditions absent from GovInfo "
      f"({len(missing) / len(status):.2%} of attempts)")
if len(missing):
    print(missing.groupby(["year", "kind"]).size().to_string())